# Experimento 15 v3 — implementação com Ket

Esta versão substitui PennyLane e Qiskit por [Ket](https://quantumket.org/). Os kernels quânticos são fidelidades de estados, $K(x,y)=|\langle\phi(x)|\phi(y)\rangle|^2$, calculadas a partir de estados preparados pelo simulador denso do Ket.

O notebook executa os cinco experimentos completos:

1. SVR clássico com kernel RBF;
2. Double-angle entangled feature map;
3. Fourier feature map;
4. Pauli feature map Y/YY padrão;
5. Pauli feature map Y/YY avançado com data map de Riemann.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ket import CNOT, H, P, RX, RY, Process, dump
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVR

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_DIR = PROJECT_ROOT / "dataset"
RESULTS_DIR = PROJECT_ROOT / "results" / "main_v2"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RUN_EXPERIMENTS = True
MAX_FEATURES = 30
RANDOM_SEED = 42

In [ ]:
dataset = pd.read_csv(DATASET_DIR / "riemann_features.csv")
feature_ranking = pd.read_csv(
    PROJECT_ROOT / "results" / "data_analysis" / "feature_ranking_full.csv"
).sort_values("avg_rank", kind="stable")
features = feature_ranking["Unnamed: 0"].tolist()[:MAX_FEATURES]

X = dataset.drop(columns=["distance"])
y = dataset["distance"]

X = X[:2000]
y = y[:2000]

split = int(0.8 * len(X))
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print("Treino:", X_train.shape, y_train.shape)
print("Teste: ", X_test.shape, y_test.shape)

## Utilitários de estado e kernel no Ket

In [ ]:
def ket_state_vector(feature_map, x):
    """Prepara |phi(x)> no Ket e devolve o vetor de estado denso."""
    x = np.asarray(x, dtype=float)
    n_qubits = len(x)
    process = Process(simulator="dense", num_qubits=n_qubits)
    qubits = process.alloc(n_qubits)
    feature_map(x, qubits)

    sparse_state = dump(qubits).states
    state = np.zeros(2**n_qubits, dtype=np.complex128)
    for basis_state, amplitude in sparse_state.items():
        state[basis_state] = amplitude
    return state


def ket_state_matrix(X_values, feature_map):
    X_values = np.asarray(X_values, dtype=float)
    return np.vstack([ket_state_vector(feature_map, row) for row in X_values])


def ket_fidelity_kernel(X_left, X_right, feature_map):
    """Matriz K_ij = |<phi(X_left_i)|phi(X_right_j)>|^2."""
    X_left = np.asarray(X_left, dtype=float)
    X_right = np.asarray(X_right, dtype=float)

    states_left = ket_state_matrix(X_left, feature_map)
    if X_left.shape == X_right.shape and np.array_equal(X_left, X_right):
        states_right = states_left
    else:
        states_right = ket_state_matrix(X_right, feature_map)

    overlaps = states_left.conj() @ states_right.T
    kernel = np.abs(overlaps) ** 2
    return np.clip(kernel.real, 0.0, 1.0)


def make_ket_kernel(feature_map):
    return lambda A, B: ket_fidelity_kernel(A, B, feature_map)

In [ ]:
def ket_state_matrix(X_values, feature_map):
    """Prepara e normaliza uma matriz de estados (Ket pode usar float32 internamente)."""
    X_values = np.asarray(X_values, dtype=float)
    states = np.vstack([ket_state_vector(feature_map, row) for row in X_values])
    norms = np.linalg.norm(states, axis=1, keepdims=True)
    if np.any(norms == 0):
        raise RuntimeError("O Ket devolveu um estado de norma zero.")
    return states / norms

## Mapas de características em Ket

In [ ]:
_ROTATIONS = {"X": RX, "Y": RY}


def biaxal_entangled_feature_map(
    x, qubits, rotation_first="X", rotation_second="Y"
):
    """RX(x), anel de CNOTs e RY(x), equivalente ao mapa original."""
    first = _ROTATIONS[rotation_first.upper()]
    second = _ROTATIONS[rotation_second.upper()]

    for angle, qubit in zip(x, qubits):
        first(float(angle), qubit)

    for i in range(len(qubits) - 1):
        CNOT(qubits[i], qubits[i + 1])
    if len(qubits) > 2:
        CNOT(qubits[-1], qubits[0])

    for angle, qubit in zip(x, qubits):
        second(float(angle), qubit)


def spectral_harmonic_feature_map(x, qubits):
    for angle, qubit in zip(x, qubits):
        RX(float(angle), qubit)
        P(float(angle), qubit)  # RZ e P diferem apenas por uma fase global.

    for i in range(len(qubits) - 1):
        CNOT(qubits[i], qubits[i + 1])

    for angle, qubit in zip(x, qubits):
        P(float(np.sin(angle) + np.cos(2 * angle)), qubit)


def zeta_phase_data_map(values):
    values = np.asarray(values, dtype=float)
    safe_values = values + 1e-8
    theta_like = np.sum(safe_values * np.log(safe_values))
    value = np.cos(theta_like)

    for i in range(len(values) - 1):
        value += 0.3 * np.cos(values[i] - values[i + 1])
        value += 0.3 * np.sin(values[i] * values[i + 1])
    return float(value)


def _default_pauli_data_map(values):
    values = np.asarray(values, dtype=float)
    if len(values) == 1:
        return float(values[0])
    return float(np.prod(np.pi - values))


def _apply_y_evolution(phase_angle, qubit):
    RX(np.pi / 2, qubit)
    P(float(phase_angle), qubit)
    RX(-np.pi / 2, qubit)


def _apply_yy_evolution(phase_angle, qubit_a, qubit_b):
    RX(np.pi / 2, qubit_a)
    RX(np.pi / 2, qubit_b)
    CNOT(qubit_a, qubit_b)
    P(float(phase_angle), qubit_b)
    CNOT(qubit_a, qubit_b)
    RX(-np.pi / 2, qubit_a)
    RX(-np.pi / 2, qubit_b)


def pauli_y_yy_feature_map(x, qubits, reps=2, data_map_func=None):
    """Mapa Pauli Y/YY full-entanglement, alpha=2, implementado em Ket."""
    mapper = data_map_func or _default_pauli_data_map

    for _ in range(reps):
        H(qubits)

        for i, qubit in enumerate(qubits):
            _apply_y_evolution(2.0 * mapper(x[[i]]), qubit)

        for i in range(len(qubits)):
            for j in range(i + 1, len(qubits)):
                _apply_yy_evolution(
                    2.0 * mapper(x[[i, j]]), qubits[i], qubits[j]
                )


def pauli_default_feature_map(x, qubits):
    pauli_y_yy_feature_map(x, qubits, reps=2)


def zeta_phase_feature_map(x, qubits):
    pauli_y_yy_feature_map(
        x, qubits, reps=2, data_map_func=zeta_phase_data_map
    )


def preproc_rbf_s1(X_values):
    return np.exp(-(np.asarray(X_values) ** 2) / 2.0)

### Inspeção dos circuitos gerados pelo Ket

`Process.gates()` fornece a representação nativa do circuito sem recorrer ao desenhador do Qiskit.

In [ ]:
def inspect_ket_circuit(feature_map, x):
    x = np.asarray(x, dtype=float)
    process = Process(simulator="dense", num_qubits=len(x))
    qubits = process.alloc(len(x))
    feature_map(x, qubits)
    return process.gates()

rng = np.random.default_rng(RANDOM_SEED)
x_example = rng.uniform(0, np.pi, 4)

print("Double-angle map:")
display(inspect_ket_circuit(biaxal_entangled_feature_map, x_example))
print("Fourier map:")
display(inspect_ket_circuit(spectral_harmonic_feature_map, x_example))

## SVR clássico

In [ ]:
def _metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return mse, np.sqrt(mse), r2_score(y_true, y_pred)


def run_classic_rbf_experiment():
    results = []
    param_grid = {
        "svr__C": [0.1, 1, 5, 10],
        "svr__epsilon": [0.001, 0.01, 0.1, 0.5],
        "svr__gamma": ["scale", 0.01, 0.1, 1.0],
    }

    for n in range(2, MAX_FEATURES + 1):
        print(f"SVR clássico com {n} características")
        pipeline = Pipeline([
            ("scaler", MinMaxScaler()),
            ("svr", SVR(kernel="rbf")),
        ])
        grid = GridSearchCV(
            pipeline, param_grid, scoring="neg_mean_squared_error",
            cv=TimeSeriesSplit(n_splits=2), n_jobs=-1,
        )
        grid.fit(X_train[features[:n]].to_numpy(), y_train)
        y_pred = grid.best_estimator_.predict(X_test[features[:n]].to_numpy())
        mse, rmse, r2 = _metrics(y_test, y_pred)
        results.append({
            "N Features": n, "MSE": mse, "RMSE": rmse, "R2": r2,
            "Best C": grid.best_params_["svr__C"],
            "Best epsilon": grid.best_params_["svr__epsilon"],
            "Best gamma": grid.best_params_["svr__gamma"],
        })

    results_df = pd.DataFrame(results).sort_values("MSE").reset_index(drop=True)
    output_path = RESULTS_DIR / "classical_experiment_results.csv"
    results_df.to_csv(output_path, index=False)
    print(f"CSV salvo: {output_path}", flush=True)
    return results_df

### Executar modelo: classic_rbf

In [ ]:
if RUN_EXPERIMENTS:
    classic_rbf_results = run_classic_rbf_experiment()
    display(classic_rbf_results)

## SVR com kernels Ket: double-angle e Fourier

In [ ]:
def run_ket_callable_kernel_experiment(
    feature_map, output_filename, kernel_label
):
    results = []
    param_grid = {"svr__C": [1, 5], "svr__epsilon": [0.01, 0.1]}

    for n in range(2, MAX_FEATURES + 1):
        print(f"Kernel Ket {kernel_label} com {n} qubits")
        quantum_kernel = make_ket_kernel(feature_map)
        pipeline = Pipeline([
            ("scaler", MinMaxScaler(feature_range=(0, np.pi))),
            ("svr", SVR(kernel=quantum_kernel)),
        ])
        grid = GridSearchCV(
            pipeline, param_grid, scoring="neg_mean_squared_error",
            cv=TimeSeriesSplit(n_splits=2),
        )
        grid.fit(X_train[features[:n]].to_numpy(), y_train)
        y_pred = grid.best_estimator_.predict(X_test[features[:n]].to_numpy())
        mse, rmse, r2 = _metrics(y_test, y_pred)
        results.append({
            "kernel": f"{n}_{kernel_label}", "n_qubits": n,
            "MSE": mse, "RMSE": rmse, "R2": r2,
            "Best C": grid.best_params_["svr__C"],
            "Best epsilon": grid.best_params_["svr__epsilon"],
        })

    results_df = pd.DataFrame(results).sort_values("MSE").reset_index(drop=True)
    output_path = RESULTS_DIR / output_filename
    results_df.to_csv(output_path, index=False)
    print(f"CSV salvo: {output_path}", flush=True)
    return results_df


def run_biaxal_entangled_experiment():
    return run_ket_callable_kernel_experiment(
        biaxal_entangled_feature_map,
        "simple_experiment_results.csv", "XY",
    )


def run_spectral_harmonic_experiment():
    return run_ket_callable_kernel_experiment(
        spectral_harmonic_feature_map,
        "fourier_experiment_results.csv", "Fourier",
    )


### Executar modelo: biaxal_entangled

In [ ]:
if RUN_EXPERIMENTS:
    biaxal_entangled_results = run_biaxal_entangled_experiment()
    display(biaxal_entangled_results)

### Executar modelo: spectral_harmonic

In [ ]:
if RUN_EXPERIMENTS:
    spectral_harmonic_results = run_spectral_harmonic_experiment()
    display(spectral_harmonic_results)

## SVR com mapas Pauli Y/YY em Ket

In [ ]:
def run_quantum_experiment_pauli_feature_map(
    feature_map, output_filename, label
):
    results = []
    param_grid = {
        "C": [0.1, 1, 5, 10],
        "epsilon": [0.001, 0.01, 0.1, 0.5],
    }

    for n in range(2, MAX_FEATURES + 1):
        print(f"Mapa Pauli Ket {label} com {n} características")
        X_train_subset = X_train[features[:n]].to_numpy()
        X_test_subset = X_test[features[:n]].to_numpy()

        scaler = MinMaxScaler(feature_range=(0, np.pi))
        X_train_scaled = preproc_rbf_s1(scaler.fit_transform(X_train_subset))
        X_test_scaled = preproc_rbf_s1(scaler.transform(X_test_subset))

        train_states = ket_state_matrix(X_train_scaled, feature_map)
        test_states = ket_state_matrix(X_test_scaled, feature_map)
        kernel_train = np.abs(train_states.conj() @ train_states.T) ** 2
        kernel_test = np.abs(test_states.conj() @ train_states.T) ** 2

        grid = GridSearchCV(
            SVR(kernel="precomputed"), param_grid,
            scoring="neg_mean_squared_error",
            cv=TimeSeriesSplit(n_splits=2), n_jobs=-1,
        )
        grid.fit(kernel_train, y_train)
        y_pred = grid.best_estimator_.predict(kernel_test)
        mse, rmse, r2 = _metrics(y_test, y_pred)
        results.append({
            "N Features": n, "MSE": mse, "RMSE": rmse, "R2": r2,
            "Best C": grid.best_params_["C"],
            "Best epsilon": grid.best_params_["epsilon"],
        })

    results_df = pd.DataFrame(results).sort_values("MSE").reset_index(drop=True)
    output_path = RESULTS_DIR / output_filename
    results_df.to_csv(output_path, index=False)
    print(f"CSV salvo: {output_path}", flush=True)
    return results_df


def run_pauli_default_experiment():
    return run_quantum_experiment_pauli_feature_map(
        pauli_default_feature_map,
        "quantum_pauli_results_default.csv", "padrão",
    )


def run_zeta_phase_experiment():
    return run_quantum_experiment_pauli_feature_map(
        zeta_phase_feature_map,
        "quantum_pauli_results.csv", "Riemann",
    )


### Executar modelo: pauli_default

In [ ]:
if RUN_EXPERIMENTS:
    pauli_default_results = run_pauli_default_experiment()
    display(pauli_default_results)

### Executar modelo: zeta_phase

In [ ]:
if RUN_EXPERIMENTS:
    zeta_phase_results = run_zeta_phase_experiment()
    display(zeta_phase_results)

## Geração dos gráficos finais

In [ ]:
RESULT_FILES = {
    "Gaussian RBF": ("classical_experiment_results.csv", "N Features"),
    "Zeta-Phase Pauli (Ket)": ("quantum_pauli_results.csv", "N Features"),
    "Standard Pauli (Ket)": ("quantum_pauli_results_default.csv", "N Features"),
    "Biaxial Entangled (Ket)": ("simple_experiment_results.csv", "n_qubits"),
    "Spectral Harmonic (Ket)": ("fourier_experiment_results.csv", "n_qubits"),
}
COLORS = ["b", "r", "c", "g", "m"]
MARKERS = ["o", "s", "P", "^", "D"]


def plot_combined_results(metric):
    loaded = {}
    for label, (filename, x_column) in RESULT_FILES.items():
        path = RESULTS_DIR / filename
        if not path.exists():
            raise FileNotFoundError(
                f"Resultado ausente: {path}. Execute os experimentos primeiro."
            )
        frame = pd.read_csv(path)
        if metric == "RMSE" and "RMSE" not in frame and "MSE" in frame:
            frame["RMSE"] = np.sqrt(frame["MSE"])
        if metric == "R2" and "R2" not in frame and "r2" in frame:
            frame["R2"] = frame["r2"]
        if x_column not in frame and x_column == "N Features" and "Features" in frame:
            frame["N Features"] = frame["Features"]
        loaded[label] = (frame.sort_values(x_column), x_column)

    fig, axes = plt.subplots(1, 5, figsize=(22, 5), sharey=True)
    for ax, (label, (frame, x_column)), color in zip(axes, loaded.items(), COLORS):
        ax.plot(frame[x_column], frame[metric], color + "-o")
        ax.set_title(label)
        ax.set_xlabel("Número de características/qubits")
        ax.set_xticks(range(2, MAX_FEATURES + 1, 2))
        ax.grid(True)
    axes[0].set_ylabel(metric)
    fig.suptitle(f"{metric} por número de características/qubits", fontsize=14)
    fig.tight_layout()
    suffix = "" if metric == "RMSE" else "_r2"
    fig.savefig(RESULTS_DIR / f"combined_results_subplots{suffix}.png", dpi=160, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(10, 6))
    for (label, (frame, x_column)), color, marker in zip(loaded.items(), COLORS, MARKERS):
        plt.plot(frame[x_column], frame[metric], color=color, marker=marker, label=label)
    plt.xlabel("Número de características/qubits")
    plt.ylabel(metric)
    plt.title(f"Comparação de {metric} entre os mapas de características")
    plt.xticks(range(2, MAX_FEATURES + 1, 2))
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"combined_results{suffix}.png", dpi=160, bbox_inches="tight")
    plt.show()


if RUN_EXPERIMENTS:
    plot_combined_results("RMSE")
    plot_combined_results("R2")